# Generate environment

By joining different sources of data:
- Weather data for TMY
- Electricity price data
- Thermal load data

And save them as a unified dataset.

In [17]:
from pathlib import Path
import pandas as pd
import datetime
from loguru import logger

%load_ext autoreload
%autoreload 2

data_path: Path = Path("../../data/datasets/partial")
data_output_path: Path = Path("../../data/datasets")
output_path: Path = Path("../results/visualizations")

weather_filename: str = "tmy_2005_Borg_El_Arab.h5" # guadix tabernas
electricity_filename: str = "electricity_price_data_20211231_20241231.h5"
load_filename: str = "thermal_load_data_med_20220101_20241231.h5" # "thermal_load_data_20220101_20241231.h5" # "thermal_load_data_20220101_20241231.h5"
water_filename: str = "water_data_med_20220101_20241231.h5" # "water_data_psa_20220101_20241231.h5"

environment_id: str = "_med_" # "_andasol_"

year_start: int = 2022
year_end: int = 2024
constraint_Tv_to_Tamb: bool = False


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
# Load datasets
df_weather = pd.read_hdf(data_path / weather_filename) if weather_filename.endswith("h5") else pd.read_csv(data_path / weather_filename, index_col=0, parse_dates=True)
df_electricity = pd.read_hdf(data_path / electricity_filename) if electricity_filename.endswith("h5") else pd.read_csv(data_path / electricity_filename, index_col=0, parse_dates=True)
df_load = pd.read_hdf(data_path / load_filename) if load_filename.endswith("h5") else pd.read_csv(data_path / load_filename, index_col=0, parse_dates=True)
df_water = pd.read_hdf(data_path / water_filename) if water_filename.endswith("h5") else pd.read_csv(data_path / water_filename, index_col=0, parse_dates=True)

# Create a pandas series with one-hour resolution
start_date = datetime.datetime(year_start, 1, 1, 0, 0, 0, tzinfo=datetime.timezone.utc)
end_date = datetime.datetime(year_end, 12, 31, 23, 0, 0, tzinfo=datetime.timezone.utc)
date_rng = pd.date_range(start=start_date, end=end_date, freq='h', tz='UTC', inclusive="both")

display(df_weather.head())
display(df_electricity.head())
display(df_load.head())
display(df_water.head())


,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,Td
time,,,,,,,,,,
2005-01-01 00:00:00+00:00,0,0,0,0,13.3,84,0.0,-142.6,0.5,10.7
2005-01-01 01:00:00+00:00,0,0,0,0,13.2,72,0.0,-107.2,0.0,8.2
2005-01-01 02:00:00+00:00,0,0,0,0,13.0,74,0.0,-94.4,0.0,8.4
2005-01-01 03:00:00+00:00,0,0,0,0,12.9,77,0.0,-86.5,0.0,8.9
2005-01-01 04:00:00+00:00,0,0,0,0,12.8,81,0.0,-80.1,0.0,9.7


,Ce_pvpc_eur_MWh,Ce_spot_market_price_eur_MWh,Ce_pvpc_eur_kWh,Ce_spot_market_price_eur_kWh
datetime,,,,
2021-12-31 23:00:00+00:00,204.51,145.86,0.20451,0.14586
2022-01-01 00:00:00+00:00,171.35,114.90,0.17135,0.11490
2022-01-01 01:00:00+00:00,172.70,113.87,0.17270,0.11387
2022-01-01 02:00:00+00:00,156.07,97.80,0.15607,0.09780
2022-01-01 03:00:00+00:00,159.08,97.80,0.15908,0.09780


,Q_kW,Tv_C,mv_kgh,md_m3h
2022-01-01 00:00:00+00:00,0.0,0.0,0.0,0.0
2022-01-01 01:00:00+00:00,0.0,0.0,0.0,0.0
2022-01-01 02:00:00+00:00,0.0,0.0,0.0,0.0
2022-01-01 03:00:00+00:00,0.0,0.0,0.0,0.0
2022-01-01 04:00:00+00:00,0.0,0.0,0.0,0.0


,water_price_morocco_eur_m3,water_price_morocco_alternative_eur_m3,water_price_egypt_eur_m3,water_price_egypt_alternative_eur_m3,water_price_morocco_eur_l,water_price_morocco_alternative_eur_l,water_price_egypt_eur_l,water_price_egypt_alternative_eur_l,Vavail_m3,Vavail_l
2022-01-01 00:00:00+00:00,0.029054,2.2980,0.060471,2.2980,0.000029,0.002298,0.00006,0.002298,1.2,1200.0
2022-01-01 01:00:00+00:00,0.029054,2.2774,0.060471,2.2774,0.000029,0.002277,0.00006,0.002277,1.2,1200.0
2022-01-01 02:00:00+00:00,0.029054,1.9560,0.060471,1.9560,0.000029,0.001956,0.00006,0.001956,1.2,1200.0
2022-01-01 03:00:00+00:00,0.029054,1.9560,0.060471,1.9560,0.000029,0.001956,0.00006,0.001956,1.2,1200.0
2022-01-01 04:00:00+00:00,0.029054,1.9148,0.060471,1.9148,0.000029,0.001915,0.00006,0.001915,1.2,1200.0


In [19]:
# Electricity price and thermal load data should be available for the selected period
# Raise an error if the data is not available
if df_electricity.index[-1] < end_date:
    logger.warning(f"Electricity price data is not long enough: {df_electricity.index[-1]} < {end_date}. Filling difference")
    df_electricity = pd.concat([df_electricity, pd.Series(index=pd.date_range(start=df_electricity.index[-1] + datetime.timedelta(hours=1), end=end_date, freq='h', tz='UTC'))])
    df_electricity = df_electricity.ffill()
    
assert df_electricity.index[0] <= start_date and df_electricity.index[-1] >= end_date, f"Electricity price data is not available for the selected period: {df_electricity.index[0]} - {df_electricity.index[-1]}"
assert df_load.index[0] <= start_date and df_load.index[-1] >= end_date, f"Thermal load data is not available for the selected period: {df_load.index[0]} - {df_load.index[-1]}"

# Weather data needs to be adapted to the selected period (modify its year)
# Repeat data for each year
# if df_weather.index[0] <= start_date and df_weather.index[-1] >= end_date:
df_weather_ = df_weather.copy()
df_weather_ = pd.concat([df_weather_] * (year_end - year_start + 1))
df_weather_ = df_weather_.set_index(date_rng[:len(df_weather_)]).reindex(date_rng).ffill()

# Check if the weather data is available for the selected period
assert df_weather_.index[0] <= start_date and df_weather_.index[-1] >= end_date, f"Weather data is not available for the selected period: {df_weather.index[0]} - {df_weather.index[-1]}"
df_weather = df_weather_

if constraint_Tv_to_Tamb:
    # Modify Tv so that Tv always satisfies the condition Tv < Tamb - 5
    violations = df_load[df_load['Tv_C'] > df_weather['Tamb_C']+ 5 ]

    # Check if any violations exist
    if not violations.empty:
        logger.warning(f"Found {len(violations)} violations of Tv+5<Tamb")

    df_load['Tv_C'] = df_load['Tv_C'].where(df_load['Tv_C'] >= df_weather['Tamb_C'] + 5, df_weather['Tamb_C'] + 5)



2025-04-04 07:07:39.003 | WARNING  | __main__:<module>:4 - Electricity price data is not long enough: 2024-12-31 22:00:00+00:00 < 2024-12-31 23:00:00+00:00. Filling difference


In [20]:
# Join datasets
df = pd.concat([df_weather, df_electricity, df_load, df_water], axis=1, join="inner")

logger.info(f"Resulting dataframe has {len(df)} rows")

# Save to hdf5 and csv
fname = f"environment_data{environment_id}{df.index[0].strftime('%Y%m%d')}_{df.index[-1].strftime('%Y%m%d')}"
df.sort_index(inplace=True)
df.to_hdf(data_output_path / f"{fname}.h5", key="data", mode="w", format="table")
df.to_csv(data_output_path / f"{fname}.csv")
    
logger.info(f"Saved {fname} to {data_output_path}")
df


2025-04-04 07:07:42.097 | INFO     | __main__:<module>:4 - Resulting dataframe has 26304 rows


2025-04-04 07:07:42.743 | INFO     | __main__:<module>:12 - Saved environment_data_med_20220101_20241231 to ../../data/datasets


,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,Td,...,water_price_morocco_eur_m3,water_price_morocco_alternative_eur_m3,water_price_egypt_eur_m3,water_price_egypt_alternative_eur_m3,water_price_morocco_eur_l,water_price_morocco_alternative_eur_l,water_price_egypt_eur_l,water_price_egypt_alternative_eur_l,Vavail_m3,Vavail_l
2022-01-01 00:00:00+00:00,0.0,0.0,0.0,0.0,13.3,84.0,0.0,-142.6,0.5,10.7,...,0.029054,2.2980,0.060471,2.2980,0.000029,0.002298,0.00006,0.002298,1.200000,1200.000000
2022-01-01 01:00:00+00:00,0.0,0.0,0.0,0.0,13.2,72.0,0.0,-107.2,0.0,8.2,...,0.029054,2.2774,0.060471,2.2774,0.000029,0.002277,0.00006,0.002277,1.200000,1200.000000
2022-01-01 02:00:00+00:00,0.0,0.0,0.0,0.0,13.0,74.0,0.0,-94.4,0.0,8.4,...,0.029054,1.9560,0.060471,1.9560,0.000029,0.001956,0.00006,0.001956,1.200000,1200.000000
2022-01-01 03:00:00+00:00,0.0,0.0,0.0,0.0,12.9,77.0,0.0,-86.5,0.0,8.9,...,0.029054,1.9560,0.060471,1.9560,0.000029,0.001956,0.00006,0.001956,1.200000,1200.000000
2022-01-01 04:00:00+00:00,0.0,0.0,0.0,0.0,12.8,81.0,0.0,-80.1,0.0,9.7,...,0.029054,1.9148,0.060471,1.9148,0.000029,0.001915,0.00006,0.001915,1.200000,1200.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 19:00:00+00:00,0.0,0.0,0.0,0.0,8.2,100.0,0.0,133.3,0.0,8.2,...,0.029054,3.1220,0.060471,3.1220,0.000029,0.003122,0.00006,0.003122,1.019608,1019.607843
2024-12-31 20:00:00+00:00,0.0,0.0,0.0,0.0,8.2,100.0,0.0,133.3,0.0,8.2,...,0.029054,3.0170,0.060471,3.0170,0.000029,0.003017,0.00006,0.003017,1.019608,1019.607843
2024-12-31 21:00:00+00:00,0.0,0.0,0.0,0.0,8.2,100.0,0.0,133.3,0.0,8.2,...,0.029054,2.8880,0.060471,2.8880,0.000029,0.002888,0.00006,0.002888,1.019608,1019.607843
2024-12-31 22:00:00+00:00,0.0,0.0,0.0,0.0,8.2,100.0,0.0,133.3,0.0,8.2,...,0.029054,2.7874,0.060471,2.7874,0.000029,0.002787,0.00006,0.002787,1.019608,1019.607843


In [21]:
df_env = pd.read_hdf(data_output_path / f"{fname}.h5")
df_env


,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,Td,...,water_price_morocco_eur_m3,water_price_morocco_alternative_eur_m3,water_price_egypt_eur_m3,water_price_egypt_alternative_eur_m3,water_price_morocco_eur_l,water_price_morocco_alternative_eur_l,water_price_egypt_eur_l,water_price_egypt_alternative_eur_l,Vavail_m3,Vavail_l
2022-01-01 00:00:00+00:00,0.0,0.0,0.0,0.0,13.3,84.0,0.0,-142.6,0.5,10.7,...,0.029054,2.2980,0.060471,2.2980,0.000029,0.002298,0.00006,0.002298,1.200000,1200.000000
2022-01-01 01:00:00+00:00,0.0,0.0,0.0,0.0,13.2,72.0,0.0,-107.2,0.0,8.2,...,0.029054,2.2774,0.060471,2.2774,0.000029,0.002277,0.00006,0.002277,1.200000,1200.000000
2022-01-01 02:00:00+00:00,0.0,0.0,0.0,0.0,13.0,74.0,0.0,-94.4,0.0,8.4,...,0.029054,1.9560,0.060471,1.9560,0.000029,0.001956,0.00006,0.001956,1.200000,1200.000000
2022-01-01 03:00:00+00:00,0.0,0.0,0.0,0.0,12.9,77.0,0.0,-86.5,0.0,8.9,...,0.029054,1.9560,0.060471,1.9560,0.000029,0.001956,0.00006,0.001956,1.200000,1200.000000
2022-01-01 04:00:00+00:00,0.0,0.0,0.0,0.0,12.8,81.0,0.0,-80.1,0.0,9.7,...,0.029054,1.9148,0.060471,1.9148,0.000029,0.001915,0.00006,0.001915,1.200000,1200.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 19:00:00+00:00,0.0,0.0,0.0,0.0,8.2,100.0,0.0,133.3,0.0,8.2,...,0.029054,3.1220,0.060471,3.1220,0.000029,0.003122,0.00006,0.003122,1.019608,1019.607843
2024-12-31 20:00:00+00:00,0.0,0.0,0.0,0.0,8.2,100.0,0.0,133.3,0.0,8.2,...,0.029054,3.0170,0.060471,3.0170,0.000029,0.003017,0.00006,0.003017,1.019608,1019.607843
2024-12-31 21:00:00+00:00,0.0,0.0,0.0,0.0,8.2,100.0,0.0,133.3,0.0,8.2,...,0.029054,2.8880,0.060471,2.8880,0.000029,0.002888,0.00006,0.002888,1.019608,1019.607843
2024-12-31 22:00:00+00:00,0.0,0.0,0.0,0.0,8.2,100.0,0.0,133.3,0.0,8.2,...,0.029054,2.7874,0.060471,2.7874,0.000029,0.002787,0.00006,0.002787,1.019608,1019.607843


In [22]:
from plotly_resampler import FigureWidgetResampler
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

var_ids: list[str] = [
    ["Tamb_C", "Tv_C"], 
    "HR_pct", "precip_mm", "Q_kW", 
    ["Ce_pvpc_eur_MWh", "Ce_spot_market_price_eur_MWh"],
    ["water_price_morocco_eur_m3", "water_price_egypt_eur_m3", "water_price_morocco_alternative_eur_m3", "water_price_egypt_alternative_eur_m3"], 
    "Vavail_m3",]
var_units: list[str] = ["°C", "%", "mm", "kW<sub>th</sub>", "€/MW<sub>e</sub>", "€/m<sup>3</sup>","m<sup>3</sup>"]

fig = make_subplots(rows=len(var_ids), cols=1, shared_xaxes=True)
fig = FigureWidgetResampler(fig)

for i, (var_id, var_unit) in enumerate(zip(var_ids, var_units)):
    var_id = [var_id] if not isinstance(var_id, list) else var_id
    
    for v_id in var_id:
        fig.add_trace(
            go.Scattergl(name=v_id, showlegend=True), 
            hf_x=df_env.index, 
            hf_y=np.ascontiguousarray( df_env[v_id] ), 
            # max_n_samples=2_000,
            row=i + 1, col=1
        )
    fig.update_yaxes(title_text=var_unit, row=i + 1)

fig.update_layout(
    height=1400,
    width=800,
    title="<b>Environment variables</b>",
    title_x=0.1,
    legend_traceorder="normal",
    legend=dict(orientation="h", y=1.15, xanchor="left", x=0),
    margin=dict(l=20, r=20, t=340, b=20),
)

fig


FigureWidgetResampler({
    'data': [{'name': '<b style="color:sandybrown">[R]</b> Tamb_C <i style="color:#fc9944">~1D</i>',
              'showlegend': True,
              'type': 'scattergl',
              'uid': '298c9901-68f5-4212-bb56-7f5c08b652e7',
              'x': array([datetime.datetime(2022, 1, 1, 0, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2022, 1, 1, 13, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2022, 1, 2, 6, 0, tzinfo=datetime.timezone.utc), ...,
                          datetime.datetime(2024, 12, 30, 6, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2024, 12, 30, 14, 0, tzinfo=datetime.timezone.utc),
                          datetime.datetime(2024, 12, 31, 23, 0, tzinfo=datetime.timezone.utc)],
                         shape=(1000,), dtype=object),
              'xaxis': 'x',
              'y': array([13.3, 21.4,  9.9, ..., 10.1, 17.1,  8.2], shape=(1000,)),
     

In [23]:
from phd_visualizations import save_figure

start, end = fig.layout.xaxis.range 
save_figure(fig, f"solhycool_env_viz{environment_id}{start[:10].replace('-', '')}_{end[:10].replace('-', '')}", 
            figure_path=output_path, formats=["png", "svg"])


2025-04-04 07:08:39.023 | INFO     | phd_visualizations:save_figure:38 - Figure saved in [PosixPath('../results/visualizations')]/solhycool_env_viz_med_20221025_20230123.png
2025-04-04 07:08:44.643 | INFO     | phd_visualizations:save_figure:38 - Figure saved in [PosixPath('../results/visualizations')]/solhycool_env_viz_med_20221025_20230123.svg
